# WSI classification dataset (feature location and label) creation

## Example with TCGA-LUAD dataset (Normal_TS, Tumor_DX1)

### Step 1. Extract masks for the TCGA_LUAD dataset 

### Step 2. Extract features for the TCGA_LUAD

### Step 3. Copy .pt files into the folders: set1 and combined set1_set2_set3

##### Coordinates also are copied accordingly into the folders Coordinates_set1 and Coordinates_scombined set1_set2_set3

In [4]:
import os
import shutil
from pathlib import Path 
import numpy as np
from tqdm import tqdm
from collections import defaultdict
import torch
# Suppress the NumPy warning if it still appears
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='torch')

def organize_features(input_folder, output_folder, set1_substring="Informative_set1"):
    """
    Reorganize feature files from WSI-based structure to set-based structure.
    
    Args:
        input_folder: Path to input folder with encoder/WSI/filename.pt structure
        output_folder: Path to output folder for reorganized files
        set1_substring: Substring to filter files for set1 (default: "Informative_set1")
    """
    input_path = Path(input_folder)
    output_path = Path(output_folder)
    
    # Get all encoders (subdirectories in input folder)
    encoders = [d.name for d in input_path.iterdir() if d.is_dir()]
    
    print(f"Found {len(encoders)} encoders: {encoders}")
    print(f"Set1 filter: Files containing '{set1_substring}'\n")
    
    # Dictionary to store files by encoder and WSI for concatenation
    # Structure: {encoder: {wsi_name: [list of file paths]}}
    files_by_encoder_wsi = defaultdict(lambda: defaultdict(list))
    
    # Dictionary to store set1 files by encoder
    # Structure: {encoder: [list of file paths containing set1_substring]}
    set1_files_by_encoder = defaultdict(list)
    
    # Collect all files organized by encoder and WSI
    for encoder in encoders:
        encoder_path = input_path / encoder
        wsi_folders = [d for d in encoder_path.iterdir() if d.is_dir()]
        
        print(f"{encoder}: Found {len(wsi_folders)} WSI folders")
        
        for wsi_folder in wsi_folders:
            wsi_name = wsi_folder.name
            pt_files = sorted(list(wsi_folder.glob("*.pt")))
            
            if pt_files:
                # Store all files for concatenation
                files_by_encoder_wsi[encoder][wsi_name] = pt_files
                
                # Filter files containing set1_substring for set1 folder
                set1_files = [f for f in pt_files if set1_substring in f.name]
                set1_files_by_encoder[encoder].extend(set1_files)
    
    # Create set1 folder - copy only files containing set1_substring
    print("\n" + "="*80)
    print(f"Creating set1 folder (files containing '{set1_substring}')...")
    print("="*80)
    
    set1_path = output_path / "Extracted_features_set1"
    
    for encoder in encoders:
        encoder_set1_path = set1_path / encoder
        encoder_set1_path.mkdir(parents=True, exist_ok=True)
            
        set1_files = set1_files_by_encoder[encoder]
            
        if not set1_files:
            print(f"{encoder}: No files containing '{set1_substring}' found!")
            continue
            
        # Copy files containing set1_substring
        for file_path in set1_files:
            # --- START OF CHANGE ---
            # Check for the specific substring and rename the destination file
            if "_Informative_set1" in file_path.name:
                # Take the part before '_Informative_set1' and add the original extension
                new_file_name = file_path.name.split("_Informative_set1")[0] + file_path.suffix
            else:
                # Keep original name if the pattern is not found
                new_file_name = file_path.name
                    
            dest_path = encoder_set1_path / new_file_name
            # --- END OF CHANGE ---
                
            shutil.copy2(file_path, dest_path)
            
        print(f"{encoder}: Copied {len(set1_files)} files containing '{set1_substring}' to set1/{encoder}/")
    
    # Create set1_set2_set3_combined folder - concatenate ALL files per WSI
    print("\n" + "="*80)
    print("Creating set1_set2_set3_combined folder (concatenated ALL files per WSI)...")
    print("="*80)
    
    combined_path = output_path / "Extracted_features_set1_set2_set3_combined"
    
    for encoder in tqdm(encoders, desc="Processing encoders"):
        encoder_combined_path = combined_path / encoder
        encoder_combined_path.mkdir(parents=True, exist_ok=True)
        
        wsi_count = 0
        error_count = 0
        
        for wsi_name, file_list in tqdm(files_by_encoder_wsi[encoder].items(), 
                                         desc=f"  {encoder}", 
                                         leave=False):
            # Load all features for this WSI (ALL files, not just set1)
            features_list = []
            for file_path in sorted(file_list):
                try:
                    feature = torch.load(file_path, map_location='cpu', weights_only=False)
                    features_list.append(feature)
                except Exception as e:
                    print(f"\nError loading {file_path}: {e}")
                    error_count += 1
                    continue
            
            if features_list:
                # Concatenate features
                try:
                    concatenated = torch.cat(features_list, dim=0)
                    
                    # Save concatenated features as WSI_name.pt
                    output_file = encoder_combined_path / f"{wsi_name}.pt"
                    torch.save(concatenated, output_file)
                    wsi_count += 1
                except Exception as e:
                    print(f"\nError concatenating features for {encoder}/{wsi_name}: {e}")
                    error_count += 1
        
        print(f"{encoder}: Created {wsi_count} concatenated WSI files" + 
              (f" ({error_count} errors)" if error_count > 0 else ""))
    
    print("\n" + "="*80)
    print("Processing complete!")
    print("="*80)
    print(f"\nOutput structure:")
    print(f"  {output_path}/set1/<encoder>/<files_with_{set1_substring}.pt>")
    print(f"  {output_path}/set1_set2_set3_combined/<encoder>/<WSI_name.pt>")

def verify_output(output_folder, set1_substring="Informative_set1"):
    """Verify the output structure and print statistics"""
    output_path = Path(output_folder)
    
    print("\n" + "="*80)
    print("VERIFICATION SUMMARY")
    print("="*80)
    
    # Check set1
    set1_path = output_path / "set1"
    if set1_path.exists():
        print(f"\nset1 folder (files containing '{set1_substring}'):")
        for encoder_dir in sorted(set1_path.iterdir()):
            if encoder_dir.is_dir():
                file_count = len(list(encoder_dir.glob("*.pt")))
                # Calculate total size
                total_size = sum(f.stat().st_size for f in encoder_dir.glob("*.pt"))
                print(f"  {encoder_dir.name}: {file_count} files ({total_size / (1024**3):.2f} GB)")
                
                # Show sample filenames to verify they contain the substring
                sample_files = list(encoder_dir.glob("*.pt"))[:3]
                if sample_files:
                    print(f"    Sample files:")
                    for f in sample_files:
                        print(f"      - {f.name}")
    
    # Check combined
    combined_path = output_path / "Extracted_features_set1_set2_set3_combined"
    if combined_path.exists():
        print("\nset1_set2_set3_combined folder (ALL files concatenated per WSI):")
        for encoder_dir in sorted(combined_path.iterdir()):
            if encoder_dir.is_dir():
                file_count = len(list(encoder_dir.glob("*.pt")))
                # Calculate total size
                total_size = sum(f.stat().st_size for f in encoder_dir.glob("*.pt"))
                print(f"  {encoder_dir.name}: {file_count} WSI files ({total_size / (1024**3):.2f} GB)")
                
                # Sample check - load one file to verify shape
                sample_files = list(encoder_dir.glob("*.pt"))
                if sample_files:
                    try:
                        sample = torch.load(sample_files[0], map_location='cpu', weights_only=False)
                        print(f"    Sample: {sample_files[0].name} - shape: {sample.shape}")
                    except Exception as e:
                        print(f"    Could not load sample: {e}")

In [5]:
def process_coordinates(input_path, output_path):
    # The specific suffixes to look for in order
    # Logic: We look for file matching: {WSI_NAME}_{SUFFIX} inside the WSI folder
    SETS_SUFFIXES = ["_Informative_set1_coordinates.npy", "_Informative_set2_coordinates.npy", "_Informative_set3_coordinates.npy"]
    input_path = Path(input_path)
    output_path = Path(output_path)

    # 1. Define the specific output directories as requested
    out_dir_set1 = output_path / "Coordinates_set1"
    out_dir_combined = output_path / "Coordinates_set1_set2_set3_combined"

    # 2. Create them
    out_dir_set1.mkdir(parents=True, exist_ok=True)
    out_dir_combined.mkdir(parents=True, exist_ok=True)

    # 3. Get list of WSI directories (exclude hidden files/root files)
    wsi_dirs = [d for d in input_path.iterdir() if d.is_dir()]
    
    print(f"Found {len(wsi_dirs)} WSI directories to process.")
    print(f"Output Directory 1: {out_dir_set1}")
    print(f"Output Directory 2: {out_dir_combined}")

    # 4. Iterate through each WSI folder
    for wsi_dir in tqdm(wsi_dirs, desc="Processing WSIs"):
        wsi_name = wsi_dir.name
        
        # --- TASK 1: Process Set 1 (Copy & Rename) ---
        # Construct the expected filename for set1
        src_file_set1 = wsi_dir / f"{wsi_name}{SETS_SUFFIXES[0]}"
        
        if src_file_set1.exists():
            # Define new name: WSI_Name_coordinates.npy
            dst_name = f"{wsi_name}_coordinates.npy"
            dst_file = out_dir_set1 / dst_name
            
            # Copy the file (faster than loading/saving numpy array)
            shutil.copy2(src_file_set1, dst_file)
        
        # --- TASK 2: Process Combined (Concatenate) ---
        coords_list = []
        found_any = False
        
        # Loop through set1 -> set2 -> set3 in order
        for suffix in SETS_SUFFIXES:
            src_file = wsi_dir / f"{wsi_name}{suffix}"
            
            if src_file.exists():
                try:
                    # Load the numpy array
                    arr = np.load(src_file)
                    coords_list.append(arr)
                    found_any = True
                except Exception as e:
                    print(f"Error loading {src_file.name}: {e}")

        if found_any:
            try:
                # Concatenate all arrays found along the 0-axis (rows)
                combined_coords = np.concatenate(coords_list, axis=0)
                
                # Save concatenated file: WSI_Name_coordinates.npy
                combined_dst_name = f"{wsi_name}_coordinates.npy"
                combined_dst_file = out_dir_combined / combined_dst_name
                
                np.save(combined_dst_file, combined_coords)
            except Exception as e:
                print(f"Error concatenating/saving for {wsi_name}: {e}")

    print("\n✓ Processing combining coordinates is complete.")

In [11]:
if __name__ == "__main__":
    # Set your paths here
    path = "/data_64T_3/Raja/MUFASA/Survival_analysis/ROI_patch/Selected_WSI_for_ROI/TCGA_BLCA"
    tissue = "MUFASA" # Normal_TS, Tumor_DX1
    input_folder_for_features = f"{path}/{tissue}/Extracted_features"
    output_folder_for_features = f"{path}/{tissue}"
    
    # You can customize the substring based on the feature file name to filter for set1
    set1_substring = "Informative_set1"  # Change this if needed
    
    # Run the organization
    organize_features(input_folder_for_features, output_folder_for_features, set1_substring)
    
    # Verify the output
    verify_output(output_folder_for_features, set1_substring)

    # Step 1: Set tissue type to "Normal_TS" and run the program
    # Step 2: Set tissue type to "Tumor_DX1" and run the program
    # Observe the folders "set1" and "set1_set2_set3_combined" in the path "/data_64T_3/Raja/Extracted_Features/TCGA_KIRC/"

    # Preparing coordinate files (if visualization is needed)
    input_folder_for_coordinates = f"{path}/{tissue}/Coordinates"
    output_folder_for_coordinates = f"{path}/{tissue}"
    process_coordinates(input_folder_for_coordinates, output_folder_for_coordinates)

Found 1 encoders: ['uni']
Set1 filter: Files containing 'Informative_set1'

uni: Found 6 WSI folders

Creating set1 folder (files containing 'Informative_set1')...
uni: Copied 6 files containing 'Informative_set1' to set1/uni/

Creating set1_set2_set3_combined folder (concatenated ALL files per WSI)...


Processing encoders: 100%|████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.61it/s]


uni: Created 6 concatenated WSI files

Processing complete!

Output structure:
  /data_64T_3/Raja/MUFASA/Survival_analysis/ROI_patch/Selected_WSI_for_ROI/TCGA_BLCA/MUFASA/set1/<encoder>/<files_with_Informative_set1.pt>
  /data_64T_3/Raja/MUFASA/Survival_analysis/ROI_patch/Selected_WSI_for_ROI/TCGA_BLCA/MUFASA/set1_set2_set3_combined/<encoder>/<WSI_name.pt>

VERIFICATION SUMMARY

set1_set2_set3_combined folder (ALL files concatenated per WSI):
  uni: 6 WSI files (0.34 GB)
    Sample: TCGA-CF-A3MF-01Z-00-DX1.91768C26-5BB1-4DE6-9737-AA26BD687A75.pt - shape: torch.Size([16734, 1024])
Found 6 WSI directories to process.
Output Directory 1: /data_64T_3/Raja/MUFASA/Survival_analysis/ROI_patch/Selected_WSI_for_ROI/TCGA_BLCA/MUFASA/Coordinates_set1
Output Directory 2: /data_64T_3/Raja/MUFASA/Survival_analysis/ROI_patch/Selected_WSI_for_ROI/TCGA_BLCA/MUFASA/Coordinates_set1_set2_set3_combined


Processing WSIs: 100%|███████████████████████████████████████████████████████████████████| 6/6 [00:00<00:00, 928.15it/s]


✓ Processing combining coordinates is complete.


##### Verifying the dimensions of features in .pt files (Optional)

In [146]:
import torch
import numpy as np
from pathlib import Path

def view_pt_file(filepath, show_data=False, max_rows=10):
    """
    View contents of a .pt file
    
    Args:
        filepath: Path to .pt file
        show_data: Whether to show actual data values (default: False)
        max_rows: Maximum rows to display if show_data=True (default: 10)
    """
    print("="*80)
    print(f"File: {filepath}")
    print("="*80)
    
    try:
        # Load the file
        data = torch.load(filepath, map_location='cpu', weights_only=False)
        
        # Determine data type
        if isinstance(data, torch.Tensor):
            print(f"\nType: PyTorch Tensor")
            print(f"Shape: {data.shape}")
            print(f"Dtype: {data.dtype}")
            print(f"Device: {data.device}")
            print(f"Requires grad: {data.requires_grad}")
            print(f"Memory: {data.element_size() * data.nelement() / (1024**2):.2f} MB")
            
            # Statistics
            if data.dtype in [torch.float32, torch.float64, torch.float16]:
                print(f"\nStatistics:")
                print(f"  Min: {data.min().item():.6f}")
                print(f"  Max: {data.max().item():.6f}")
                print(f"  Mean: {data.mean().item():.6f}")
                print(f"  Std: {data.std().item():.6f}")
            
            # Show data sample
            if show_data:
                print(f"\nData (first {max_rows} rows):")
                if len(data.shape) == 1:
                    print(data[:max_rows])
                elif len(data.shape) == 2:
                    print(data[:max_rows, :])
                else:
                    print(data.flatten()[:max_rows])
                    
        elif isinstance(data, dict):
            print(f"\nType: Dictionary")
            print(f"Keys: {list(data.keys())}")
            print(f"\nDetails:")
            for key, value in data.items():
                if isinstance(value, torch.Tensor):
                    print(f"  {key}: Tensor{tuple(value.shape)} ({value.dtype})")
                else:
                    print(f"  {key}: {type(value).__name__}")
                    
        elif isinstance(data, list):
            print(f"\nType: List")
            print(f"Length: {len(data)}")
            if data:
                print(f"First element type: {type(data[0]).__name__}")
                if isinstance(data[0], torch.Tensor):
                    print(f"First element shape: {data[0].shape}")
                    
        elif isinstance(data, np.ndarray):
            print(f"\nType: NumPy Array")
            print(f"Shape: {data.shape}")
            print(f"Dtype: {data.dtype}")
            print(f"Memory: {data.nbytes / (1024**2):.2f} MB")
            
        else:
            print(f"\nType: {type(data).__name__}")
            print(f"Value: {data}")
            
    except Exception as e:
        print(f"\nError loading file: {e}")
    
    print("="*80 + "\n")


def quick_view(filepath):
    """Quick view - just basic info"""
    try:
        data = torch.load(filepath, map_location='cpu', weights_only=False)
        if isinstance(data, torch.Tensor):
            print(f"{Path(filepath).name}: Tensor{tuple(data.shape)} {data.dtype} ({data.element_size() * data.nelement() / (1024**2):.2f} MB)")
        elif isinstance(data, dict):
            print(f"{Path(filepath).name}: Dict with keys {list(data.keys())}")
        else:
            print(f"{Path(filepath).name}: {type(data).__name__}")
    except Exception as e:
        print(f"{Path(filepath).name}: Error - {e}")


def batch_view(folder, pattern="*.pt", quick=True):
    """View multiple .pt files in a folder"""
    folder_path = Path(folder)
    files = sorted(list(folder_path.glob(pattern)))
    
    print(f"Found {len(files)} files matching '{pattern}' in {folder}\n")
    
    for file in files:
        if quick:
            quick_view(file)
        else:
            view_pt_file(file)


# Example usage
if __name__ == "__main__":
    # Single file - detailed view
    path = "/data_64T_3/Shared/Extracted_Features/MUFASA/CAMELYON16/Normal_TS/set1/conch/normal_002_Informative_set1_combined_features.pt"
    
    view_pt_file(path, show_data=True, max_rows=5) 
    
    # Quick view
    quick_view(path)  

File: /data_64T_3/Shared/Extracted_Features/MUFASA/CAMELYON16/Normal_TS/set1/conch/normal_002_Informative_set1_combined_features.pt

Type: PyTorch Tensor
Shape: torch.Size([6823, 1024])
Dtype: torch.float32
Device: cpu
Requires grad: False
Memory: 26.65 MB

Statistics:
  Min: -4.091016
  Max: 4.179572
  Mean: -0.000000
  Std: 1.000000

Data (first 5 rows):
tensor([[-1.0490, -0.6133,  1.5779,  ...,  1.0042,  0.7003,  0.1889],
        [-0.8382, -1.2484,  0.8012,  ...,  0.8234,  0.6210,  0.6934],
        [-0.6102, -0.8640,  0.5064,  ...,  0.8342,  0.6321,  0.4438],
        [-0.9474, -1.3105,  1.1417,  ...,  0.8956,  0.5512,  0.6324],
        [-0.5757, -0.7547,  0.4059,  ...,  0.7862,  0.6715,  0.3599]])

normal_002_Informative_set1_combined_features.pt: Tensor(6823, 1024) torch.float32 (26.65 MB)


### Step 4: Prepare a csv file with slide_name and label for TCGA_LUAD

In [ ]:
#!/usr/bin/env python3
"""Create slide CSV with custom Normal=0, Tumor=1 mapping for the dataset with an example TCGA_LUAD."""
import pandas as pd
from pathlib import Path
import sys

# Configuration
dataset = 'TCGA_STAD'
input_path = f'/data_64T_2/Dataset/{dataset}/images'
extensions = ['*.svs', '*.tif', '*.tiff']
tissues = ['Normal_TS', 'Tumor_DX1']

output_path = f'/data_64T_2/Dataset/{dataset}/labels/{dataset}_binary_class_label.csv'

# Exclusion list - add WSI names (without extension) to exclude.
# Because of no MPP or no mask, no features would have been extracted. so include those slide names as below.
exclude_list = [ 
    'slide_name_1',
    'slide_name_2',
    # Add more slide names here as needed
]

# Custom label mapping
label_map = {tissues[0] : 0, tissues[1] : 1}

# Generate data
data = []
for tissue in tissues:
    tissue_folder = Path(input_path) / tissue
    if not tissue_folder.exists():
        print(f"⚠️  Warning: {tissue_folder} does not exist, skipping...")
        continue
    
    # Check if it's a directory structure or files directly in tissue folder
    svs_files = [f for ext in extensions for f in tissue_folder.glob(ext)]
    
    # print(f"tissue_folder: {tissue_folder}")
    # print(f"len(svs_files) : {len(svs_files)}")

    if svs_files:
        # Files are directly in the tissue folder
        label = label_map[tissue]
        for image in sorted(svs_files):
            if image.stem not in exclude_list:
                data.append({'slide_name': image.stem, 'label': label})

# # Save CSV
df = pd.DataFrame(data)
df.to_csv(output_path, index=False)

print(f"✓ Created {output_path}")
print(f"  Total slides: {len(df)}")
print(f"  Excluded slides: {len(exclude_list)}")
print(f"  Normal (0): {(df['label']==0).sum()}")
print(f"  Tumor (1): {(df['label']==1).sum()}")

### Step 5: Prepare k-fold common train-test splits to run with different MIL model

### Step 6: Replacing slide_name with its feature path from different encdoer

In [ ]:
The output code provides csv files with wsi name along its label. We need to include path of the feature file of each dataset.

In [21]:
#!/usr/bin/env python3
"""Add feature paths to WSI names in CSV files."""
import pandas as pd
from pathlib import Path
import sys

base_path = "/data_64T_3/Shared/Extracted_Features"
dataset = "TCGA_LUAD"    # TCGA_LUAD
pre_processor = "TRIDENT"   # MUFASA, TRIDENT, CLAM, HISTOLAB 
tissue = ["Normal_TS", "Tumor_DX1"]
encoder = "resnet50_1024"   # 'resnet18', 'uni'

# Configuration
csv_dir = f'/data_64T_2/Dataset/{dataset}/labels/5-fold-common-split' # path that contains split inforamation. 
# Even if there is a single csv file, place it in a folder and provide its location

# feature location
feature_dirs = [f'{base_path}/{pre_processor}/{dataset}/{tissue[0]}/Extracted_features/{encoder}',
                f'{base_path}/{pre_processor}/{dataset}/{tissue[1]}/Extracted_features/{encoder}']

# output path to store
# output_path = f"/home/rajaj/Project/7.Multi_Phase_Tile_Extractor_Experiments/WSI_Classification/MIL_BASELINE/datasets/{dataset}/{dataset}_{pre_processor}_{encoder}_splits"
output_path = f"{base_path}/{pre_processor}/{dataset}/{dataset}_{pre_processor}_{encoder}_splits"
Path(output_path).mkdir(parents=True, exist_ok=True)
# ------------------------------------------------------------------------------------------------
# Build feature lookup dictionary
print("Building feature file index...")
feature_map = {}
dirs_found = True # Flag to track if all feature directories exist

for feature_dir in feature_dirs:
    # Check if directory exists to avoid errors
    if Path(feature_dir).exists():
        for pt_file in Path(feature_dir).rglob('*.pt'):
            slide_name = pt_file.stem
            feature_map[slide_name] = str(pt_file)
    else:
        print(f"Warning: Directory not found: {feature_dir}")
        dirs_found = False # Set flag to False if any directory is missing

if not dirs_found:
    print("⚠️  Stopping: One or more feature directories are missing.")
    sys.exit(1) # Exit script if directories are bad

print(f"Found {len(feature_map)} feature files across {len(feature_dirs)} directories")

# Process all CSV files
for csv_file in Path(csv_dir).glob('*.csv'):
    df = pd.read_csv(csv_file)
    missing = []
    
    # Update all *_slide_path columns
    for col in df.columns:
        if col.endswith('_slide_path'):
            def get_path(x):
                if pd.notna(x):
                    # Ensure x is a string to match dictionary keys safely
                    x_str = str(x).strip() 
                    if x_str in feature_map:
                        return feature_map[x_str]
                    else:
                        missing.append(x_str)
                        return x  # Keep original if not found
                return x
            
            df[col] = df[col].apply(get_path)
    
    # --- CHANGED LOGIC HERE ---
    if missing:
        unique_missing = set(missing)
        print(f"⚠️  SKIPPING {csv_file.name}: Missing {len(unique_missing)} features: {unique_missing}")
        # Do NOT save the file, simply continue to the next one

    # Only reach here if missing list is empty
    output_file = f'{output_path}/{csv_file.stem}_with_path.csv'
    df.to_csv(output_file, index=False)
    
    print(f"✓ {csv_file.name} -> {Path(output_file).name}")

Building feature file index...
feature_map : 399
Found 399 feature files across 2 directories
✓ Camelyon16_binary_class_label_common_split_new.csv -> Camelyon16_binary_class_label_common_split_new_with_path.csv


### Step 7: Update the dataset location in MIL_BASELINE/configs/CLAM_MB.yaml file

In [ ]:
Update the parameters: DATASET_NAME, dataset_root_dir, log_root_dir carefully

### Step 7: Run a MIL model

### Step 8: Run test on a set of WSI

In [ ]:
Input dataset: Test.csv
slide_path	label
/path/WSI1.pt	0
/path/WSI2.pt	1

In [ ]:
python test_mil.py \
--yaml_path configs/CLAM_MB_MIL.yaml \
--test_dataset_csv datasets/Inference.csv \
--model_weight_path logs/path/Best_EPOCH_57.pth \
--test_log_dir logs/Test_output

In [ ]:
Output folder strucutre 
logs/Test_output
    |-Test_CLAM_MB_MIL.yaml
    |-Test_dataset_TCGA_LUAD.csv
    |-Test_Log_CLAM_MB_MIL.txt

### Step 9: Inference on a set of WSI that does not label

In [ ]:
python test_mil.py \
--yaml_path configs/CLAM_MB_MIL.yaml \
--test_dataset_csv datasets/Inference.csv \
--model_weight_path logs/path/Best_EPOCH_57.pth \
--test_log_dir logs/Inference_output

In [ ]:
Output folder contains a .csv file with the following data

slide_path	  slide_name	prediction	         probabilities
/path/WSI1.pt    WSI1           0	    [0.9865760803222656, 0.013423897325992584]
/path/WSI2.pt	 WSI2           1	    [0.1328595131635666, 0.867140531539917]

### Step 10: Generating slide level visualizations

##### Draw Feature Map

##### Draw Attention Map